# Project 1: Customer Churn Prediction

**Goal:** predict which customers are likely to stop using a service, and find the patterns behind churn.

**Dataset:** Telco Customer Churn (a free dataset on Kaggle: search "Telco Customer Churn"). Download the CSV and keep it in the same folder as this notebook (or upload it to Colab).

**Steps:** load and clean the data, explore it (EDA), prepare features, train classification models, evaluate with accuracy, recall and ROC-AUC, tune the best model and explain the results.

> 

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             classification_report, ConfusionMatrixDisplay, RocCurveDisplay)

try:
    from xgboost import XGBClassifier      # optional: pip install xgboost
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost is not installed, so it will be skipped.")

sns.set_theme(style="whitegrid")

## 1. Load the data

In [ ]:
DATA_PATH = "WA_Fn-UseC_-Telco-Customer-Churn.csv"   # change this if your file name is different
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
print("Rows and columns:", df.shape)
df.head()

In [ ]:
df.info()
df.describe()

## 2. Clean the data

In [ ]:
# TotalCharges is stored as text because a few rows are blank. Convert it to numbers.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Missing TotalCharges:", df["TotalCharges"].isna().sum())

# Those blank rows are customers with tenure = 0 (they have not been billed yet), so 0 is a sensible fill value.
df["TotalCharges"] = df["TotalCharges"].fillna(0)

df = df.drop_duplicates()
df = df.drop(columns=["customerID"])                    # an ID does not help prediction
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})      # target as 1 / 0
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
print("Missing values left:", df.isna().sum().sum())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print(df["Churn"].value_counts(normalize=True).round(3))
sns.countplot(x="Churn", data=df)
plt.title("Customers who churned (1) and stayed (0)")
plt.show()

In [ ]:
def churn_rate_plot(col):
    rate = df.groupby(col)["Churn"].mean().sort_values(ascending=False) * 100
    rate.plot(kind="bar", figsize=(6, 4), color="#2E6F9E")
    plt.ylabel("Churn rate (%)")
    plt.title(f"Churn rate by {col}")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

for col in ["Contract", "InternetService", "PaymentMethod", "TechSupport"]:
    churn_rate_plot(col)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(data=df, x="tenure", hue="Churn", bins=30, multiple="stack", ax=axes[0])
axes[0].set_title("Tenure (months) by churn")
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1])
axes[1].set_title("Monthly charges by churn")
plt.tight_layout()
plt.show()

In [ ]:
sns.heatmap(df[["tenure", "MonthlyCharges", "TotalCharges", "Churn"]].corr(), annot=True, cmap="Blues", fmt=".2f")
plt.title("Correlation of numerical columns")
plt.show()

## 4. Prepare the data for modelling

In [ ]:
X = df.drop(columns="Churn")
y = df["Churn"]

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
print("Numerical:", num_cols)
print("Categorical:", cat_cols)

# stratify keeps the same churn share in the train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Scale the numbers, one-hot encode the categories (the split is done first to avoid data leakage)
pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
print(X_train.shape, X_test.shape)

## 5. Train and compare models

Churners are the minority class, so `class_weight="balanced"` is used to give them more importance.

In [ ]:
def make_pipeline(model):
    return Pipeline([("pre", pre), ("clf", model)])

models = {
    "Logistic Regression": make_pipeline(LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Random Forest": make_pipeline(RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)),
}
if HAS_XGB:
    models["XGBoost"] = make_pipeline(XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                                    eval_metric="logloss", random_state=42))

rows = []
for name, m in models.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    proba = m.predict_proba(X_test)[:, 1]
    rows.append({"Model": name,
                 "Accuracy": accuracy_score(y_test, pred),
                 "Precision": precision_score(y_test, pred),
                 "Recall": recall_score(y_test, pred),
                 "F1": f1_score(y_test, pred),
                 "ROC-AUC": roc_auc_score(y_test, proba)})
results = pd.DataFrame(rows).set_index("Model").round(3)
results.sort_values("ROC-AUC", ascending=False)

## 6. Evaluate the best model

In [ ]:
best_name = results["ROC-AUC"].idxmax()
best_model = models[best_name]
print("Best model by ROC-AUC:", best_name)
print(classification_report(y_test, best_model.predict(X_test)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test, cmap="Blues", ax=axes[0])
axes[0].set_title(f"Confusion matrix: {best_name}")
for name, m in models.items():
    RocCurveDisplay.from_estimator(m, X_test, y_test, name=name, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--", label="Random guess")
axes[1].set_title("ROC curves")
axes[1].legend()
plt.tight_layout()
plt.show()

## 7. Cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, m in models.items():
    scores = cross_val_score(m, X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"{name}: mean ROC-AUC = {scores.mean():.3f} (+/- {scores.std():.3f})")

## 8. Hyperparameter tuning (Random Forest)

In [ ]:
param_dist = {
    "clf__n_estimators": [100, 200, 300],
    "clf__max_depth": [None, 6, 10, 15],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 4],
}
search = RandomizedSearchCV(models["Random Forest"], param_dist, n_iter=10, cv=3,
                            scoring="roc_auc", random_state=42, n_jobs=-1)
search.fit(X_train, y_train)
print("Best parameters:", search.best_params_)
print("Best cross-validated ROC-AUC:", round(search.best_score_, 3))

tuned = search.best_estimator_
pred = tuned.predict(X_test)
proba = tuned.predict_proba(X_test)[:, 1]
print("Test accuracy:", round(accuracy_score(y_test, pred), 3))
print("Test recall  :", round(recall_score(y_test, pred), 3))
print("Test ROC-AUC :", round(roc_auc_score(y_test, proba), 3))

## 9. Which features drive churn?

In [ ]:
names = tuned.named_steps["pre"].get_feature_names_out()
importance = pd.Series(tuned.named_steps["clf"].feature_importances_, index=names)
top = importance.sort_values(ascending=False).head(10)
top.sort_values().plot(kind="barh", figsize=(7, 4.5), color="#2E6F9E")
plt.title("Top 10 most important features")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 10. Conclusion and business insights


1. The best model, and its accuracy, recall and ROC-AUC.
2. The top 3 factors that influence churn.
3. Two actions a company could take to reduce churn (for example, offers for month-to-month customers).
4. One limitation of this project (for example, the dataset is a sample, or the model can be improved).

In [ ]:
joblib.dump(tuned, "churn_model.pkl")      # optional: save the final model
print("Model saved.")